# PatchCore ViT-B/16 MAE Fine-Tuned 25% Mask (`x224`)

This notebook is the canonical training and evaluation workflow for the 25% masking-ratio MAE-style fine-tuned ViT-B/16 PatchCore experiment.

Run this notebook from top to bottom to reproduce the experiment locally. The notebook prepares the data split, fine-tunes the ViT backbone when needed, builds the PatchCore memory bank, evaluates the model, and writes outputs into the experiment-local artifact folders.

**Control flags (set in the Configuration cell):**

Artifacts written by this notebook:
- [`experiments/anomaly_detection/patchcore/vit_b16/x224/FT/MAE_25pct/artifacts/checkpoints`](experiments/anomaly_detection/patchcore/vit_b16/x224/FT/MAE_25pct/artifacts/checkpoints) for model checkpoints
- [`experiments/anomaly_detection/patchcore/vit_b16/x224/FT/MAE_25pct/artifacts/plots`](experiments/anomaly_detection/patchcore/vit_b16/x224/FT/MAE_25pct/artifacts/plots) for saved figures
- [`experiments/anomaly_detection/patchcore/vit_b16/x224/FT/MAE_25pct/artifacts/results`](experiments/anomaly_detection/patchcore/vit_b16/x224/FT/MAE_25pct/artifacts/results) for metrics, score files, and CSV outputs

## Pipeline

```
Stage 1  MAE fine-tuning and backbone loading
         Masked patch reconstruction on 40 000 normal wafers
         Only last 6 transformer blocks + LayerNorm are updated

Stage 2  PatchCore memory bank and saved-score reuse
         Patch embeddings from fine-tuned block-6 features, projected to 128-d
         600 000-patch coreset bank, k=3 nearest neighbours, top-10% patch scoring

Stage 3  Threshold calibration + evaluation
         95th percentile of tune-normal scores, no defect leakage
```


## Setup

This cell installs or checks optional notebook dependencies before the rest of the workflow runs.


In [ ]:
import importlib
import importlib.util
import subprocess
import sys
for pkg in ['timm', 'tqdm']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('timm + tqdm ready')


## Imports

This cell imports the Python packages used across training, evaluation, plotting, and artifact export.


In [ ]:
import os, gc, json, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, average_precision_score, precision_recall_curve
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_CUDA = DEVICE.type == 'cuda'
if USE_CUDA:
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')
print('Device:', DEVICE)
if USE_CUDA:
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  VRAM: {p.total_memory / 1000000000.0:.1f} GB')
from IPython.display import display, Image as IPImage


## Configuration

This cell resolves the repo root, defines experiment settings, and points all outputs to the local artifact folders.


In [ ]:
try:
    from pathlib import Path
    cwd = Path.cwd().resolve()
    candidate_roots = [cwd, *cwd.parents]
    PROJECT_ROOT = None
    for candidate in candidate_roots:
        if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
            PROJECT_ROOT = candidate
            break
    if PROJECT_ROOT is None:
        raise RuntimeError('Could not locate repo root containing src/wafer_defect and configs/')
    RETRAIN = False
    FORCE_REBUILD_SCORES = False
    FORCE_RERUN_UMAP = False
    DATA_PATH = str(PROJECT_ROOT / 'data' / 'raw' / 'LSWMD.pkl')
    IMAGE_SIZE = 224
    TRAIN_NORMAL_N = 40000
    TUNE_NORMAL_N = 5000
    TEST_NORMAL_N = 5000
    TEST_DEFECT_N = 250
    UNFREEZE_BLOCKS = 6
    FINETUNE_EPOCHS = 10
    FINETUNE_LR = 0.0001
    FINETUNE_BATCH = 64
    FINETUNE_WORKERS = 0
    MAE_PATCH_SIZE = 16
    MAE_MASK_RATIO = 0.25
    MAE_DECODER_HIDDEN = 512
    VIT_FEATURE_BLOCK = 6
    PATCH_EMBED_DIM = 128
    MEMORY_BANK_MAX = 600000
    SCORE_CHUNK = 512
    PATCHCORE_NN_K = 3
    TOPK_PATCH_RATIO = 0.1
    BANK_BATCH_SIZE = 128
    BANK_WORKERS = 0
    THRESHOLD_QUANTILE = 0.95
    EXPERIMENT_DIR = str(PROJECT_ROOT / 'experiments/anomaly_detection/patchcore/vit_b16/x224/FT/MAE_25pct')
    ARTIFACT_DIR = os.path.join(EXPERIMENT_DIR, 'artifacts')
    CHECKPOINTS_DIR = os.path.join(ARTIFACT_DIR, 'checkpoints')
    PLOTS_DIR = os.path.join(ARTIFACT_DIR, 'plots')
    RESULTS_DIR = os.path.join(ARTIFACT_DIR, 'results')
    FINETUNE_CKPT = os.path.join(CHECKPOINTS_DIR, 'vit_finetuned.pt')
    MODEL_EXPORT_PATH = os.path.join(CHECKPOINTS_DIR, 'model.pt')
    SCORES_EXPORT_PATH = os.path.join(RESULTS_DIR, 'scores.npz')
    METRICS_EXPORT_PATH = os.path.join(RESULTS_DIR, 'evaluation_metrics.json')
    SUMMARY_EXPORT_PATH = os.path.join(RESULTS_DIR, 'summary.json')
    UMAP_PNG_PATH = os.path.join(PLOTS_DIR, 'umap_test_embeddings.png')
    UMAP_CSV_PATH = os.path.join(RESULTS_DIR, 'umap_test_embeddings.csv')
    SCORE_DISTRIBUTION_PNG_PATH = os.path.join(PLOTS_DIR, 'score_distribution_test.png')
    CONFUSION_MATRIX_PNG_PATH = os.path.join(PLOTS_DIR, 'confusion_matrix_test.png')
    THRESHOLD_SELECTION_PNG_PATH = os.path.join(PLOTS_DIR, 'threshold_selection_tune.png')
    CLASSIFICATION_REPORT_PATH = os.path.join(RESULTS_DIR, 'classification_report.txt')
    DEFECT_RECALL_CSV_PATH = os.path.join(RESULTS_DIR, 'test_defect_recall.csv')
    TEST_SCORES_CSV_PATH = os.path.join(RESULTS_DIR, 'test_scores.csv')
    HISTORY_PLOT_PATH = os.path.join(PLOTS_DIR, 'finetune_history.png')
    EVAL_PLOT_PATH = os.path.join(PLOTS_DIR, 'evaluation_results.png')
    LEGACY_FINETUNE_CKPT = os.path.join(EXPERIMENT_DIR, 'vit_finetuned.pt')
    LEGACY_MODEL_EXPORT_PATH = os.path.join(EXPERIMENT_DIR, 'patchcore_trained_model.pt')
    LEGACY_METRICS_EXPORT_PATH = os.path.join(EXPERIMENT_DIR, 'evaluation_metrics.json')
    LEGACY_HISTORY_PLOT_PATH = os.path.join(EXPERIMENT_DIR, 'finetune_history.png')
    LEGACY_EVAL_PLOT_PATH = os.path.join(EXPERIMENT_DIR, 'evaluation_results.png')
    for path_str in [CHECKPOINTS_DIR, PLOTS_DIR, RESULTS_DIR]:
        os.makedirs(path_str, exist_ok=True)
    print(f'Project root  : {PROJECT_ROOT}')
    print(f'Artifact dir  : {ARTIFACT_DIR}')
    print(f'RETRAIN={RETRAIN}  FORCE_REBUILD_SCORES={FORCE_REBUILD_SCORES}  FORCE_RERUN_UMAP={FORCE_RERUN_UMAP}')
    print(f'MAE: unfreeze_blocks={UNFREEZE_BLOCKS}, epochs={FINETUNE_EPOCHS}, lr={FINETUNE_LR}, mask_ratio={MAE_MASK_RATIO}')
    print(f'PatchCore: block={VIT_FEATURE_BLOCK}, embed_dim={PATCH_EMBED_DIM}, bank_cap={MEMORY_BANK_MAX:,}')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "from pathlib import path\ncwd = path.cwd().resolve()\ncandidate_roots = [cwd, *cwd.parents]\nproject_root = none\nfor candidate in candidate_roots:\n    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():\n        project_root = candidate\n        break\nif project_root is none:\n    raise runtimeerror('could not locate repo root containing src/wafer_defect and configs/')\nretrain = false\nforce_rebuild_scores = false\nforce_rerun_umap = false\ndata_path = str(project_root / 'data' / 'raw' / 'lswmd.pkl')\nimage_size = 224\ntrain_normal_n = 40000\ntune_normal_n = 5000\ntest_normal_n = 5000\ntest_defect_n = 250\nunfreeze_blocks = 6\nfinetune_epochs = 10\nfinetune_lr = 0.0001\nfinetune_batch = 64\nfinetune_workers = 0\nmae_patch_size = 16\nmae_mask_ratio = 0.25\nmae_decoder_hidden = 512\nvit_feature_block = 6\npatch_embed_dim = 128\nmemory_bank_max = 600000\nscore_chunk = 512\npatchcore_nn_k = 3\ntopk_patch_ratio = 0.1\nbank_batch_size = 128\nbank_workers = 0\nthreshold_quantile = 0.95\nexperiment_dir = str(project_root / 'experiments/anomaly_detection/patchcore/vit_b16/x224/ft/mae_25pct')\nartifact_dir = os.path.join(experiment_dir, 'artifacts')\ncheckpoints_dir = os.path.join(artifact_dir, 'checkpoints')\nplots_dir = os.path.join(artifact_dir, 'plots')\nresults_dir = os.path.join(artifact_dir, 'results')\nfinetune_ckpt = os.path.join(checkpoints_dir, 'vit_finetuned.pt')\nmodel_export_path = os.path.join(checkpoints_dir, 'model.pt')\nscores_export_path = os.path.join(results_dir, 'scores.npz')\nmetrics_export_path = os.path.join(results_dir, 'evaluation_metrics.json')\nsummary_export_path = os.path.join(results_dir, 'summary.json')\numap_png_path = os.path.join(plots_dir, 'umap_test_embeddings.png')\numap_csv_path = os.path.join(results_dir, 'umap_test_embeddings.csv')\nscore_distribution_png_path = os.path.join(plots_dir, 'score_distribution_test.png')\nconfusion_matrix_png_path = os.path.join(plots_dir, 'confusion_matrix_test.png')\nthreshold_selection_png_path = os.path.join(plots_dir, 'threshold_selection_tune.png')\nclassification_report_path = os.path.join(results_dir, 'classification_report.txt')\ndefect_recall_csv_path = os.path.join(results_dir, 'test_defect_recall.csv')\ntest_scores_csv_path = os.path.join(results_dir, 'test_scores.csv')\nhistory_plot_path = os.path.join(plots_dir, 'finetune_history.png')\neval_plot_path = os.path.join(plots_dir, 'evaluation_results.png')\nlegacy_finetune_ckpt = os.path.join(experiment_dir, 'vit_finetuned.pt')\nlegacy_model_export_path = os.path.join(experiment_dir, 'patchcore_trained_model.pt')\nlegacy_metrics_export_path = os.path.join(experiment_dir, 'evaluation_metrics.json')\nlegacy_history_plot_path = os.path.join(experiment_dir, 'finetune_history.png')\nlegacy_eval_plot_path = os.path.join(experiment_dir, 'evaluation_results.png')\nfor path_str in [checkpoints_dir, plots_dir, results_dir]:\n    os.makedirs(path_str, exist_ok=true)\nprint(f'project root  : {project_root}')\nprint(f'artifact dir  : {artifact_dir}')\nprint(f'retrain={retrain}  force_rebuild_scores={force_rebuild_scores}  force_rerun_umap={force_rerun_umap}')\nprint(f'mae: unfreeze_blocks={unfreeze_blocks}, epochs={finetune_epochs}, lr={finetune_lr}, mask_ratio={mae_mask_ratio}')\nprint(f'patchcore: block={vit_feature_block}, embed_dim={patch_embed_dim}, bank_cap={memory_bank_max:,}')\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Load Dataset

This cell loads the legacy WM-811K pickle, cleans labels, and prepares the dataframe used for the experiment split.


In [ ]:
df = pd.read_pickle(DATA_PATH)
print('Raw shape:', df.shape)

def parse_failure_label(v):
    if v is None:
        return 'unknown'
    if isinstance(v, float) and np.isnan(v):
        return 'unknown'
    if isinstance(v, (list, tuple, np.ndarray)):
        a = np.array(v).reshape(-1)
        return 'unknown' if len(a) == 0 else str(a[0])
    return str(v)
df = df.copy()
df['failure_label'] = df['failureType'].apply(parse_failure_label).str.strip()
invalid = {'0', 'unknown', 'nan', 'None', '[]'}
df = df[~df['failure_label'].isin(invalid)].copy()
df['is_anomaly'] = (df['failure_label'].str.lower() != 'none').astype(int)
normal_df = df[df['is_anomaly'] == 0].copy()
defect_df = df[df['is_anomaly'] == 1].copy()
print(f'Labeled: {len(df):,}   Normal: {len(normal_df):,}   Defect: {len(defect_df):,}')
print('\nDefect breakdown:')
print(defect_df['failure_label'].value_counts())


## Split

This cell assigns fixed-seed splits for training, threshold calibration, and evaluation.


In [ ]:
req_n = TRAIN_NORMAL_N + TUNE_NORMAL_N + TEST_NORMAL_N
req_d = TEST_DEFECT_N
if len(normal_df) < req_n:
    raise ValueError(f'Need {req_n:,} normals, have {len(normal_df):,}')
if len(defect_df) < req_d:
    raise ValueError(f'Need {req_d:,} defects, have {len(defect_df):,}')
rng = np.random.default_rng(SEED)
ns = normal_df.iloc[rng.permutation(len(normal_df))].reset_index(drop=True)
ds = defect_df.iloc[rng.permutation(len(defect_df))].reset_index(drop=True)
a = TRAIN_NORMAL_N
b = a + TUNE_NORMAL_N
c = b + TEST_NORMAL_N
train_normal_df = ns.iloc[0:a].copy()
tune_normal_df = ns.iloc[a:b].copy()
test_normal_df = ns.iloc[b:c].copy()
test_defect_df = ds.iloc[0:TEST_DEFECT_N].copy()
del df, normal_df, defect_df, ns, ds
gc.collect()
print(f'Train normal : {len(train_normal_df):>7,}  (MAE fine-tuning + memory bank)')
print(f'Tune  normal : {len(tune_normal_df):>7,}  (threshold calibration, no defect leakage)')
print(f'Test  normal : {len(test_normal_df):>7,}')
print(f'Test  defect : {len(test_defect_df):>7,}')
print('\nDefect classes in test set:')
print(test_defect_df['failure_label'].value_counts())


## Dataset and MAE Helpers

This cell defines the lazy `WaferDataset` and the patch masking utilities used during MAE fine-tuning.


In [ ]:
class WaferDataset(Dataset):
    """Lazy dataset: stores raw numpy maps, converts to tensors per-batch."""

    def __init__(self, frame: pd.DataFrame, size: int=224):
        self.maps = frame['waferMap'].values
        self.labels = frame['is_anomaly'].values.astype(np.int64)
        self.size = size

    def __len__(self):
        return len(self.maps)

    def __getitem__(self, idx):
        arr = np.clip(np.array(self.maps[idx], dtype=np.int64), 0, 2)
        x = torch.tensor(arr, dtype=torch.long)
        x = F.one_hot(x, num_classes=3).permute(2, 0, 1).float()
        x = F.interpolate(x.unsqueeze(0), size=(self.size, self.size), mode='nearest').squeeze(0)
        return (x, int(self.labels[idx]))

def images_to_patches(x: torch.Tensor, patch_size: int=16) -> torch.Tensor:
    """[B, C, H, W] -> [B, N, C*patch_size*patch_size]"""
    B, C, H, W = x.shape
    h = H // patch_size
    w = W // patch_size
    patches = x.reshape(B, C, h, patch_size, w, patch_size)
    patches = patches.permute(0, 2, 4, 1, 3, 5).reshape(B, h * w, C * patch_size * patch_size)
    return patches

def apply_random_patch_mask(x: torch.Tensor, mask_ratio: float=0.25, patch_size: int=16):
    """Zeroes random patches and returns (masked_x, mask[True=masked])."""
    B, C, H, W = x.shape
    h = H // patch_size
    w = W // patch_size
    n = h * w
    n_mask = max(1, int(round(mask_ratio * n)))
    mask = torch.zeros(B, n, dtype=torch.bool, device=x.device)
    for b in range(B):
        idx = torch.randperm(n, device=x.device)[:n_mask]
        mask[b, idx] = True
    patch_mask = mask.reshape(B, h, w).unsqueeze(1).repeat(1, C, 1, 1).float()
    patch_mask = F.interpolate(patch_mask, size=(H, W), mode='nearest')
    x_masked = x * (1.0 - patch_mask)
    return (x_masked, mask)
base_ds = WaferDataset(train_normal_df, IMAGE_SIZE)
xb, yb = base_ds[0]
print(f'Sample shape: {tuple(xb.shape)}  dtype={xb.dtype}  label={yb}')


## Stage 1 - MAE Fine-Tuning

This cell defines the 25% masked patch reconstruction model and loads the saved fine-tuned backbone from the notebook artifacts when available.

Only the **last `UNFREEZE_BLOCKS` transformer blocks + final LayerNorm** are updated. The rest stay frozen to control VRAM and preserve pretrained semantics.


In [ ]:
class MaskedPatchReconstructionViT(nn.Module):
    """
    ViT-B/16 with a masked patch reconstruction head.
    Only the last UNFREEZE_BLOCKS transformer blocks + final LayerNorm are trainable.
    """

    def __init__(self, unfreeze_blocks=6, patch_size=16, in_chans=3, decoder_hidden=512):
        super().__init__()
        self.patch_size = patch_size
        self.vit = timm.create_model('vit_base_patch16_224.augreg_in21k_ft_in1k', pretrained=True, num_classes=0)
        for p in self.vit.parameters():
            p.requires_grad = False
        total_blocks = len(self.vit.blocks)
        unfreeze_from = total_blocks - unfreeze_blocks
        for block in self.vit.blocks[unfreeze_from:]:
            for p in block.parameters():
                p.requires_grad = True
        for p in self.vit.norm.parameters():
            p.requires_grad = True
        embed_dim = self.vit.embed_dim
        patch_dim = in_chans * patch_size * patch_size
        self.recon_head = nn.Sequential(nn.Linear(embed_dim, decoder_hidden), nn.GELU(), nn.Linear(decoder_hidden, patch_dim))
        trainable = sum((p.numel() for p in self.parameters() if p.requires_grad))
        total = sum((p.numel() for p in self.parameters()))
        print(f'Backbone: ViT-B/16 ({total / 1000000.0:.1f}M params)')
        print(f'Trainable: {trainable / 1000000.0:.1f}M params ({100 * trainable / total:.1f}%)')
        print(f'Frozen:    {(total - trainable) / 1000000.0:.1f}M params')

    def forward(self, x):
        feats = self.vit.forward_features(x)
        patch_tokens = feats[:, 1:, :]
        patch_recon = self.recon_head(patch_tokens)
        return patch_recon
mae_model = MaskedPatchReconstructionViT(unfreeze_blocks=UNFREEZE_BLOCKS, patch_size=MAE_PATCH_SIZE, decoder_hidden=MAE_DECODER_HIDDEN).to(DEVICE)


In [ ]:
try:
    history_loss = None
    if RETRAIN:
        rot_loader = DataLoader(WaferDataset(train_normal_df, IMAGE_SIZE), batch_size=FINETUNE_BATCH, shuffle=True, num_workers=FINETUNE_WORKERS, pin_memory=USE_CUDA, persistent_workers=FINETUNE_WORKERS > 0)
        optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, mae_model.parameters()), lr=FINETUNE_LR, weight_decay=0.0001)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FINETUNE_EPOCHS * len(rot_loader))
        scaler = torch.amp.GradScaler(enabled=USE_CUDA)
        history_loss = []
        print(f'MAE fine-tuning: {FINETUNE_EPOCHS} epochs ({len(rot_loader)} batches/epoch)..')
        for epoch in range(1, FINETUNE_EPOCHS + 1):
            mae_model.train()
            total_loss, total_samples = (0.0, 0)
            pbar = tqdm(rot_loader, desc=f'Epoch {epoch}/{FINETUNE_EPOCHS}', leave=False)
            for xb, _ in pbar:
                xb = xb.to(DEVICE)
                xb_masked, patch_mask = apply_random_patch_mask(xb, mask_ratio=MAE_MASK_RATIO, patch_size=MAE_PATCH_SIZE)
                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_CUDA):
                    pred_patches = mae_model(xb_masked)
                    tgt_patches = images_to_patches(xb, patch_size=MAE_PATCH_SIZE)
                    per_patch_mse = ((pred_patches - tgt_patches) ** 2).mean(dim=-1)
                    mask_float = patch_mask.float()
                    loss = (per_patch_mse * mask_float).sum() / mask_float.sum().clamp(min=1.0)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(mae_model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                bs = len(xb)
                total_samples += bs
                total_loss += loss.item() * bs
                pbar.set_postfix(masked_mse=f'{loss.item():.4f}')
            epoch_loss = total_loss / total_samples
            history_loss.append(epoch_loss)
            print(f'Epoch {epoch:02d}/{FINETUNE_EPOCHS}  masked_mse={epoch_loss:.6f}')
        torch.save({'vit_state_dict': mae_model.vit.state_dict(), 'training_history': history_loss, 'config': dict(backbone='vit_base_patch16_224.augreg_in21k_ft_in1k', unfreeze_blocks=UNFREEZE_BLOCKS, epochs=FINETUNE_EPOCHS, lr=FINETUNE_LR, mask_ratio=MAE_MASK_RATIO, decoder_hidden=MAE_DECODER_HIDDEN)}, FINETUNE_CKPT)
        print(f'\nFine-tuned backbone saved -> {FINETUNE_CKPT}')
    else:
        load_finetune_ckpt = FINETUNE_CKPT
        if not os.path.exists(load_finetune_ckpt) and os.path.exists(LEGACY_FINETUNE_CKPT):
            load_finetune_ckpt = LEGACY_FINETUNE_CKPT
            print(f'Using legacy fine-tuned checkpoint -> {load_finetune_ckpt}')
        if not os.path.exists(load_finetune_ckpt):
            raise FileNotFoundError(f'No saved checkpoint at {FINETUNE_CKPT} or {LEGACY_FINETUNE_CKPT}.\nMAE fine-tuning can be re-run if needed.')
        ckpt = torch.load(load_finetune_ckpt, map_location='cpu')
        if isinstance(ckpt, dict) and 'vit_state_dict' in ckpt:
            mae_model.vit.load_state_dict(ckpt['vit_state_dict'])
            history_loss = ckpt.get('training_history', None)
        else:
            mae_model.vit.load_state_dict(ckpt)
            history_loss = None
        print(f'Loaded fine-tuned backbone from: {load_finetune_ckpt}')
        if history_loss is not None:
            print(f'Training history loaded: {len(history_loss)} epochs, final loss={history_loss[-1]:.6f}')
        else:
            print('No training history in checkpoint (saved without history).')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "history_loss = none\nif retrain:\n    rot_loader = dataloader(waferdataset(train_normal_df, image_size), batch_size=finetune_batch, shuffle=true, num_workers=finetune_workers, pin_memory=use_cuda, persistent_workers=finetune_workers > 0)\n    optimizer = torch.optim.adamw(filter(lambda p: p.requires_grad, mae_model.parameters()), lr=finetune_lr, weight_decay=0.0001)\n    scheduler = torch.optim.lr_scheduler.cosineannealinglr(optimizer, t_max=finetune_epochs * len(rot_loader))\n    scaler = torch.amp.gradscaler(enabled=use_cuda)\n    history_loss = []\n    print(f'mae fine-tuning: {finetune_epochs} epochs ({len(rot_loader)} batches/epoch)..')\n    for epoch in range(1, finetune_epochs + 1):\n        mae_model.train()\n        total_loss, total_samples = (0.0, 0)\n        pbar = tqdm(rot_loader, desc=f'epoch {epoch}/{finetune_epochs}', leave=false)\n        for xb, _ in pbar:\n            xb = xb.to(device)\n            xb_masked, patch_mask = apply_random_patch_mask(xb, mask_ratio=mae_mask_ratio, patch_size=mae_patch_size)\n            optimizer.zero_grad(set_to_none=true)\n            with torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=use_cuda):\n                pred_patches = mae_model(xb_masked)\n                tgt_patches = images_to_patches(xb, patch_size=mae_patch_size)\n                per_patch_mse = ((pred_patches - tgt_patches) ** 2).mean(dim=-1)\n                mask_float = patch_mask.float()\n                loss = (per_patch_mse * mask_float).sum() / mask_float.sum().clamp(min=1.0)\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(mae_model.parameters(), max_norm=1.0)\n            scaler.step(optimizer)\n            scaler.update()\n            scheduler.step()\n            bs = len(xb)\n            total_samples += bs\n            total_loss += loss.item() * bs\n            pbar.set_postfix(masked_mse=f'{loss.item():.4f}')\n        epoch_loss = total_loss / total_samples\n        history_loss.append(epoch_loss)\n        print(f'epoch {epoch:02d}/{finetune_epochs}  masked_mse={epoch_loss:.6f}')\n    torch.save({'vit_state_dict': mae_model.vit.state_dict(), 'training_history': history_loss, 'config': dict(backbone='vit_base_patch16_224.augreg_in21k_ft_in1k', unfreeze_blocks=unfreeze_blocks, epochs=finetune_epochs, lr=finetune_lr, mask_ratio=mae_mask_ratio, decoder_hidden=mae_decoder_hidden)}, finetune_ckpt)\n    print(f'\\nfine-tuned backbone saved -> {finetune_ckpt}')\nelse:\n    load_finetune_ckpt = finetune_ckpt\n    if not os.path.exists(load_finetune_ckpt) and os.path.exists(legacy_finetune_ckpt):\n        load_finetune_ckpt = legacy_finetune_ckpt\n        print(f'using legacy fine-tuned checkpoint -> {load_finetune_ckpt}')\n    if not os.path.exists(load_finetune_ckpt):\n        raise filenotfounderror(f'no saved checkpoint at {finetune_ckpt} or {legacy_finetune_ckpt}.\\nmae fine-tuning can be re-run if needed.')\n    ckpt = torch.load(load_finetune_ckpt, map_location='cpu')\n    if isinstance(ckpt, dict) and 'vit_state_dict' in ckpt:\n        mae_model.vit.load_state_dict(ckpt['vit_state_dict'])\n        history_loss = ckpt.get('training_history', none)\n    else:\n        mae_model.vit.load_state_dict(ckpt)\n        history_loss = none\n    print(f'loaded fine-tuned backbone from: {load_finetune_ckpt}')\n    if history_loss is not none:\n        print(f'training history loaded: {len(history_loss)} epochs, final loss={history_loss[-1]:.6f}')\n    else:\n        print('no training history in checkpoint (saved without history).')\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Training History

This cell plots the MAE masked-patch MSE loss curve. When `RETRAIN=False`, the history is loaded from the checkpoint if available.


In [ ]:
try:
    if history_loss is not None:
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(range(1, len(history_loss) + 1), history_loss, marker='o', color='steelblue')
        ax.set_title('MAE Fine-Tuning Loss (Masked Patch Reconstruction)')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Masked-Patch MSE')
        plt.tight_layout()
        plt.savefig(HISTORY_PLOT_PATH, dpi=120, bbox_inches='tight')
        plt.show()
        print(f'History plot saved -> {HISTORY_PLOT_PATH}')
    elif os.path.exists(HISTORY_PLOT_PATH):
        print(f'No history in checkpoint. Displaying saved history plot: {HISTORY_PLOT_PATH}')
        display(IPImage(filename=HISTORY_PLOT_PATH))
    else:
        print('No training history available. No training history available in the saved artifacts.')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "if history_loss is not none:\n    fig, ax = plt.subplots(figsize=(7, 4))\n    ax.plot(range(1, len(history_loss) + 1), history_loss, marker='o', color='steelblue')\n    ax.set_title('mae fine-tuning loss (masked patch reconstruction)')\n    ax.set_xlabel('epoch')\n    ax.set_ylabel('masked-patch mse')\n    plt.tight_layout()\n    plt.savefig(history_plot_path, dpi=120, bbox_inches='tight')\n    plt.show()\n    print(f'history plot saved -> {history_plot_path}')\nelif os.path.exists(history_plot_path):\n    print(f'no history in checkpoint. displaying saved history plot: {history_plot_path}')\n    display(ipimage(filename=history_plot_path))\nelse:\n    print('no training history available. no training history available in the saved artifacts.')\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Stage 2 - PatchCore Memory Bank

This cell defines the PatchCore feature extractor using the fine-tuned backbone and reuses saved anomaly scores when local artifacts are available.


In [ ]:
class FineTunedPatchExtractor(nn.Module):
    """Hooks block[VIT_FEATURE_BLOCK] for spatial patch tokens."""

    def __init__(self, finetuned_vit: nn.Module, block_idx: int=VIT_FEATURE_BLOCK, proj_dim: int=PATCH_EMBED_DIM):
        super().__init__()
        self.vit = finetuned_vit
        self._feat = None
        self.vit.blocks[block_idx].register_forward_hook(lambda m, i, o: setattr(self, '_feat', o))
        self.proj = nn.Linear(self.vit.embed_dim, proj_dim, bias=False)

    def forward(self, x):
        self.vit(x)
        return self._feat[:, 1:, :]
extractor = FineTunedPatchExtractor(mae_model.vit).to(DEVICE).eval()
for p in extractor.parameters():
    p.requires_grad = False
with torch.inference_mode():
    dummy = torch.zeros(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    out = extractor(dummy)
    proj = extractor.proj(out)
print(f'Block-{VIT_FEATURE_BLOCK} output : {tuple(out.shape)}')
print(f'After projection      : {tuple(proj.shape)}')


In [ ]:
try:
    REQUIRED_SCORE_KEYS = {'tune_normal_scores', 'test_normal_scores', 'test_defect_scores', 'threshold'}
    saved_score_keys = set()
    if os.path.exists(SCORES_EXPORT_PATH):
        with np.load(SCORES_EXPORT_PATH) as _f:
            saved_score_keys = set(_f.files)
    HAS_SAVED_SCORES = REQUIRED_SCORE_KEYS.issubset(saved_score_keys)
    REUSE_SCORES = HAS_SAVED_SCORES and (not FORCE_REBUILD_SCORES)
    memory_bank = None
    if REUSE_SCORES:
        with np.load(SCORES_EXPORT_PATH) as d:
            tune_normal_scores = d['tune_normal_scores']
            test_normal_scores = d['test_normal_scores']
            test_defect_scores = d['test_defect_scores']
            best_thresh = float(d['threshold'])
        print(f'Loaded saved scores from: {SCORES_EXPORT_PATH}')
        print(f'Loaded threshold: {best_thresh:.6f}')
    else:
        loader_kw = dict(batch_size=BANK_BATCH_SIZE, shuffle=False, num_workers=BANK_WORKERS, pin_memory=USE_CUDA, persistent_workers=BANK_WORKERS > 0)
        train_loader = DataLoader(WaferDataset(train_normal_df, IMAGE_SIZE), **loader_kw)
        tune_normal_loader = DataLoader(WaferDataset(tune_normal_df, IMAGE_SIZE), **loader_kw)
        test_normal_loader = DataLoader(WaferDataset(test_normal_df, IMAGE_SIZE), **loader_kw)
        test_defect_loader = DataLoader(WaferDataset(test_defect_df, IMAGE_SIZE), **loader_kw)

        def extract_embeddings(xb: torch.Tensor) -> torch.Tensor:
            """L2-normalised patch embeddings: [B*196, proj_dim] on GPU."""
            with torch.inference_mode():
                with torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_CUDA):
                    feat = extractor(xb)
                    emb = extractor.proj(feat)
                emb = emb.float().reshape(-1, emb.shape[-1])
                emb = F.normalize(emb, p=2, dim=1)
            return emb
        sampled, total_seen, sample_ratio = ([], 0, None)
        print('Building memory bank from fine-tuned features..')
        bank_iter = tqdm(enumerate(train_loader), total=len(train_loader), desc='Bank build', unit='batch')
        for step, (xb, _) in bank_iter:
            xb = xb.to(DEVICE)
            emb = extract_embeddings(xb)
            total_seen += len(emb)
            if sample_ratio is None:
                tokens_per_img = len(emb) // len(xb)
                estimated_total = tokens_per_img * len(train_normal_df)
                sample_ratio = min(1.0, MEMORY_BANK_MAX / estimated_total)
                print(f'  Tokens/image : {tokens_per_img}')
                print(f'  Est. total   : {estimated_total:,}')
                print(f'  Sample ratio : {sample_ratio:.5f}')
            if sample_ratio < 1.0:
                k = max(1, int(round(len(emb) * sample_ratio)))
                idx = torch.randperm(len(emb), device=DEVICE)[:k]
                emb = emb[idx]
            sampled.append(emb)
            if (step + 1) % 20 == 0 or step + 1 == len(train_loader):
                n = sum((len(e) for e in sampled))
                bank_iter.set_postfix(bank_tokens=f'{n:,}')
        memory_bank = torch.cat(sampled, dim=0)
        del sampled
        gc.collect()
        if len(memory_bank) > MEMORY_BANK_MAX:
            idx = torch.randperm(len(memory_bank), device=DEVICE)[:MEMORY_BANK_MAX]
            memory_bank = memory_bank[idx]
        vram_mb = memory_bank.element_size() * memory_bank.nelement() / 1000000.0
        print(f'Final bank : {len(memory_bank):,} x {memory_bank.shape[1]}-d  ({vram_mb:.1f} MB VRAM)')

        @torch.inference_mode()
        def score_loader(loader: DataLoader, desc: str='Scoring') -> np.ndarray:
            scores = []
            for xb, _ in tqdm(loader, desc=desc, leave=False):
                xb = xb.to(DEVICE)
                emb = extract_embeddings(xb)
                B = len(xb)
                P = emb.shape[0] // B
                patch_dists = torch.empty(B * P, dtype=torch.float32, device=DEVICE)
                for start in range(0, B * P, SCORE_CHUNK):
                    end = min(start + SCORE_CHUNK, B * P)
                    chunk = emb[start:end]
                    dists = torch.cdist(chunk, memory_bank)
                    knn = dists.topk(PATCHCORE_NN_K, dim=1, largest=False).values
                    patch_dists[start:end] = knn.mean(dim=1)
                patch_dists = patch_dists.view(B, P)
                topk = max(1, int(P * TOPK_PATCH_RATIO))
                img_scores = patch_dists.topk(topk, dim=1).values.mean(dim=1)
                scores.extend(img_scores.cpu().tolist())
            return np.array(scores)
        print('Scoring splits..')
        tune_normal_scores = score_loader(tune_normal_loader, 'Score tune-normal')
        test_normal_scores = score_loader(test_normal_loader, 'Score test-normal')
        test_defect_scores = score_loader(test_defect_loader, 'Score test-defect')
        best_thresh = float(np.quantile(tune_normal_scores, THRESHOLD_QUANTILE))
        print(f'Threshold (q={THRESHOLD_QUANTILE:.2f} on tune-normal): {best_thresh:.6f}')
        np.savez_compressed(SCORES_EXPORT_PATH, tune_normal_scores=tune_normal_scores, test_normal_scores=test_normal_scores, test_defect_scores=test_defect_scores, threshold=np.array(best_thresh))
        print(f'Scores saved -> {SCORES_EXPORT_PATH}')
    print(f'\ntune_normal: mean={tune_normal_scores.mean():.4f}  std={tune_normal_scores.std():.4f}')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = 'required_score_keys = {\'tune_normal_scores\', \'test_normal_scores\', \'test_defect_scores\', \'threshold\'}\nsaved_score_keys = set()\nif os.path.exists(scores_export_path):\n    with np.load(scores_export_path) as _f:\n        saved_score_keys = set(_f.files)\nhas_saved_scores = required_score_keys.issubset(saved_score_keys)\nreuse_scores = has_saved_scores and (not force_rebuild_scores)\nmemory_bank = none\nif reuse_scores:\n    with np.load(scores_export_path) as d:\n        tune_normal_scores = d[\'tune_normal_scores\']\n        test_normal_scores = d[\'test_normal_scores\']\n        test_defect_scores = d[\'test_defect_scores\']\n        best_thresh = float(d[\'threshold\'])\n    print(f\'loaded saved scores from: {scores_export_path}\')\n    print(f\'loaded threshold: {best_thresh:.6f}\')\nelse:\n    loader_kw = dict(batch_size=bank_batch_size, shuffle=false, num_workers=bank_workers, pin_memory=use_cuda, persistent_workers=bank_workers > 0)\n    train_loader = dataloader(waferdataset(train_normal_df, image_size), **loader_kw)\n    tune_normal_loader = dataloader(waferdataset(tune_normal_df, image_size), **loader_kw)\n    test_normal_loader = dataloader(waferdataset(test_normal_df, image_size), **loader_kw)\n    test_defect_loader = dataloader(waferdataset(test_defect_df, image_size), **loader_kw)\n\n    def extract_embeddings(xb: torch.tensor) -> torch.tensor:\n        """l2-normalised patch embeddings: [b*196, proj_dim] on gpu."""\n        with torch.inference_mode():\n            with torch.amp.autocast(device_type=\'cuda\', dtype=torch.float16, enabled=use_cuda):\n                feat = extractor(xb)\n                emb = extractor.proj(feat)\n            emb = emb.float().reshape(-1, emb.shape[-1])\n            emb = f.normalize(emb, p=2, dim=1)\n        return emb\n    sampled, total_seen, sample_ratio = ([], 0, none)\n    print(\'building memory bank from fine-tuned features..\')\n    bank_iter = tqdm(enumerate(train_loader), total=len(train_loader), desc=\'bank build\', unit=\'batch\')\n    for step, (xb, _) in bank_iter:\n        xb = xb.to(device)\n        emb = extract_embeddings(xb)\n        total_seen += len(emb)\n        if sample_ratio is none:\n            tokens_per_img = len(emb) // len(xb)\n            estimated_total = tokens_per_img * len(train_normal_df)\n            sample_ratio = min(1.0, memory_bank_max / estimated_total)\n            print(f\'  tokens/image : {tokens_per_img}\')\n            print(f\'  est. total   : {estimated_total:,}\')\n            print(f\'  sample ratio : {sample_ratio:.5f}\')\n        if sample_ratio < 1.0:\n            k = max(1, int(round(len(emb) * sample_ratio)))\n            idx = torch.randperm(len(emb), device=device)[:k]\n            emb = emb[idx]\n        sampled.append(emb)\n        if (step + 1) % 20 == 0 or step + 1 == len(train_loader):\n            n = sum((len(e) for e in sampled))\n            bank_iter.set_postfix(bank_tokens=f\'{n:,}\')\n    memory_bank = torch.cat(sampled, dim=0)\n    del sampled\n    gc.collect()\n    if len(memory_bank) > memory_bank_max:\n        idx = torch.randperm(len(memory_bank), device=device)[:memory_bank_max]\n        memory_bank = memory_bank[idx]\n    vram_mb = memory_bank.element_size() * memory_bank.nelement() / 1000000.0\n    print(f\'final bank : {len(memory_bank):,} x {memory_bank.shape[1]}-d  ({vram_mb:.1f} mb vram)\')\n\n    @torch.inference_mode()\n    def score_loader(loader: dataloader, desc: str=\'scoring\') -> np.ndarray:\n        scores = []\n        for xb, _ in tqdm(loader, desc=desc, leave=false):\n            xb = xb.to(device)\n            emb = extract_embeddings(xb)\n            b = len(xb)\n            p = emb.shape[0] // b\n            patch_dists = torch.empty(b * p, dtype=torch.float32, device=device)\n            for start in range(0, b * p, score_chunk):\n                end = min(start + score_chunk, b * p)\n                chunk = emb[start:end]\n                dists = torch.cdist(chunk, memory_bank)\n                knn = dists.topk(patchcore_nn_k, dim=1, largest=false).values\n                patch_dists'
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Threshold Calibration

The decision threshold is the **95th percentile of tune-normal scores** - a fixed normal-only quantile rule that requires no defect labels and leaks no test information.


In [ ]:
try:
    tune_mean = tune_normal_scores.mean()
    tune_std = tune_normal_scores.std()
    best_fpr = float((tune_normal_scores > best_thresh).mean())
    print(f'Normal score distribution: mean={tune_mean:.4f}  std={tune_std:.4f}')
    print(f'Threshold quantile : q={THRESHOLD_QUANTILE:.2f} on tune-normal')
    print(f'Threshold value    : {best_thresh:.6f}')
    print(f'FPR on tune-normal : {best_fpr:.4f}')
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(tune_normal_scores, bins=50, alpha=0.7, color='steelblue', label='Tune normal')
    ax.axvline(best_thresh, color='red', linestyle='--', lw=2, label=f'Threshold={best_thresh:.4f} (q={THRESHOLD_QUANTILE:.2f})')
    ax.set_title('Tune-Normal Score Distribution')
    ax.set_xlabel('Anomaly Score')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'threshold_selection_tune.png'), dpi=140, bbox_inches='tight')
    plt.show()
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "tune_mean = tune_normal_scores.mean()\ntune_std = tune_normal_scores.std()\nbest_fpr = float((tune_normal_scores > best_thresh).mean())\nprint(f'normal score distribution: mean={tune_mean:.4f}  std={tune_std:.4f}')\nprint(f'threshold quantile : q={threshold_quantile:.2f} on tune-normal')\nprint(f'threshold value    : {best_thresh:.6f}')\nprint(f'fpr on tune-normal : {best_fpr:.4f}')\nfig, ax = plt.subplots(figsize=(8, 4))\nax.hist(tune_normal_scores, bins=50, alpha=0.7, color='steelblue', label='tune normal')\nax.axvline(best_thresh, color='red', linestyle='--', lw=2, label=f'threshold={best_thresh:.4f} (q={threshold_quantile:.2f})')\nax.set_title('tune-normal score distribution')\nax.set_xlabel('anomaly score')\nax.set_ylabel('count')\nax.legend(fontsize=9)\nplt.tight_layout()\nplt.savefig(os.path.join(plots_dir, 'threshold_selection_tune.png'), dpi=140, bbox_inches='tight')\nplt.show()\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Evaluate

This cell computes overall and per-class metrics on the held-out test split using the calibrated threshold.


In [ ]:
try:
    all_scores = np.concatenate([test_normal_scores, test_defect_scores])
    all_labels = np.concatenate([np.zeros(len(test_normal_scores), dtype=int), np.ones(len(test_defect_scores), dtype=int)])
    predictions = (all_scores > best_thresh).astype(int)
    roc_auc = roc_auc_score(all_labels, all_scores)
    auprc = average_precision_score(all_labels, all_scores)
    fpr_arr, tpr_arr, _ = roc_curve(all_labels, all_scores)
    precision_arr, recall_arr, _ = precision_recall_curve(all_labels, all_scores)
    print('-- Overall Test Results ----------------------')
    print(f'ROC-AUC  : {roc_auc:.4f}')
    print(f'AUPRC    : {auprc:.4f}')
    print(f'Threshold: {best_thresh:.6f}  (q={THRESHOLD_QUANTILE:.2f} on tune-normal)')
    print()
    print(classification_report(all_labels, predictions, target_names=['Normal', 'Defect'], digits=4))
    defect_class_labels = test_defect_df['failure_label'].values
    defect_preds = (test_defect_scores > best_thresh).astype(int)
    print('-- Per-class defect detection recall ----------------------------')
    print(f"  {'Defect type':<14}  {'N':>5}  {'Detected':>8}  {'Recall':>7}  {'Mean score':>10}")
    print('  ' + '-' * 52)
    perclass_results = {}
    for cls in sorted(np.unique(defect_class_labels)):
        mask = defect_class_labels == cls
        n = mask.sum()
        detected = defect_preds[mask].sum()
        recall = detected / n
        mean_score = test_defect_scores[mask].mean()
        perclass_results[cls] = {'n': int(n), 'detected': int(detected), 'recall': float(recall), 'mean_score': float(mean_score)}
        print(f'  {cls:<14}  {n:>5}  {detected:>8}  {recall:>6.1%}  {mean_score:>10.4f}')
    overall_defect_recall = defect_preds.sum() / len(defect_preds)
    print('  ' + '-' * 52)
    print(f"  {'ALL DEFECTS':<14}  {len(defect_preds):>5}  {defect_preds.sum():>8}  {overall_defect_recall:>6.1%}")
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = 'all_scores = np.concatenate([test_normal_scores, test_defect_scores])\nall_labels = np.concatenate([np.zeros(len(test_normal_scores), dtype=int), np.ones(len(test_defect_scores), dtype=int)])\npredictions = (all_scores > best_thresh).astype(int)\nroc_auc = roc_auc_score(all_labels, all_scores)\nauprc = average_precision_score(all_labels, all_scores)\nfpr_arr, tpr_arr, _ = roc_curve(all_labels, all_scores)\nprecision_arr, recall_arr, _ = precision_recall_curve(all_labels, all_scores)\nprint(\'-- overall test results ----------------------\')\nprint(f\'roc-auc  : {roc_auc:.4f}\')\nprint(f\'auprc    : {auprc:.4f}\')\nprint(f\'threshold: {best_thresh:.6f}  (q={threshold_quantile:.2f} on tune-normal)\')\nprint()\nprint(classification_report(all_labels, predictions, target_names=[\'normal\', \'defect\'], digits=4))\ndefect_class_labels = test_defect_df[\'failure_label\'].values\ndefect_preds = (test_defect_scores > best_thresh).astype(int)\nprint(\'-- per-class defect detection recall ----------------------------\')\nprint(f"  {\'defect type\':<14}  {\'n\':>5}  {\'detected\':>8}  {\'recall\':>7}  {\'mean score\':>10}")\nprint(\'  \' + \'-\' * 52)\nperclass_results = {}\nfor cls in sorted(np.unique(defect_class_labels)):\n    mask = defect_class_labels == cls\n    n = mask.sum()\n    detected = defect_preds[mask].sum()\n    recall = detected / n\n    mean_score = test_defect_scores[mask].mean()\n    perclass_results[cls] = {\'n\': int(n), \'detected\': int(detected), \'recall\': float(recall), \'mean_score\': float(mean_score)}\n    print(f\'  {cls:<14}  {n:>5}  {detected:>8}  {recall:>6.1%}  {mean_score:>10.4f}\')\noverall_defect_recall = defect_preds.sum() / len(defect_preds)\nprint(\'  \' + \'-\' * 52)\nprint(f"  {\'all defects\':<14}  {len(defect_preds):>5}  {defect_preds.sum():>8}  {overall_defect_recall:>6.1%}")\n'
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


In [ ]:
try:
    fig = plt.figure(figsize=(21, 10))
    gs = fig.add_gridspec(2, 4, hspace=0.4, wspace=0.35)
    ax_dist = fig.add_subplot(gs[0, 0])
    ax_roc = fig.add_subplot(gs[0, 1])
    ax_pr = fig.add_subplot(gs[0, 2])
    ax_cm = fig.add_subplot(gs[0, 3])
    ax_pc = fig.add_subplot(gs[1, :])
    ax_dist.hist(test_normal_scores, bins=50, alpha=0.7, color='steelblue', label='Normal')
    ax_dist.hist(test_defect_scores, bins=50, alpha=0.7, color='tomato', label='Defect')
    ax_dist.axvline(best_thresh, color='black', linestyle='--', label=f'Threshold={best_thresh:.4f}')
    ax_dist.set_title('Anomaly Score Distributions')
    ax_dist.set_xlabel('Anomaly Score')
    ax_dist.set_ylabel('Count')
    ax_dist.legend(fontsize=9)
    ax_roc.plot(fpr_arr, tpr_arr, color='darkorange', lw=2, label=f'ROC-AUC = {roc_auc:.4f}')
    ax_roc.plot([0, 1], [0, 1], 'k--', alpha=0.4)
    op_fpr = (test_normal_scores > best_thresh).mean()
    op_tpr = (test_defect_scores > best_thresh).mean()
    ax_roc.scatter([op_fpr], [op_tpr], color='red', zorder=5, label=f'Operating point\nFPR={op_fpr:.3f} TPR={op_tpr:.3f}')
    ax_roc.set_title('ROC Curve')
    ax_roc.set_xlabel('False Positive Rate')
    ax_roc.set_ylabel('True Positive Rate')
    ax_roc.legend(fontsize=9)
    baseline = all_labels.mean()
    ax_pr.plot(recall_arr, precision_arr, color='purple', lw=2, label=f'AUPRC = {auprc:.4f}')
    ax_pr.axhline(baseline, color='gray', linestyle='--', lw=1, label=f'No-skill baseline = {baseline:.3f}')
    ax_pr.set_title('Precision-Recall Curve')
    ax_pr.set_xlabel('Recall')
    ax_pr.set_ylabel('Precision')
    ax_pr.set_xlim(0, 1)
    ax_pr.set_ylim(0, 1.02)
    ax_pr.legend(fontsize=9)
    cm = confusion_matrix(all_labels, predictions)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred Normal', 'Pred Defect'], yticklabels=['True Normal', 'True Defect'], ax=ax_cm)
    ax_cm.set_title('Confusion Matrix')
    classes = list(perclass_results.keys())
    recalls = [perclass_results[c]['recall'] for c in classes]
    counts = [perclass_results[c]['n'] for c in classes]
    mean_scores = [perclass_results[c]['mean_score'] for c in classes]
    order = np.argsort(recalls)
    classes = [classes[i] for i in order]
    recalls = [recalls[i] for i in order]
    counts = [counts[i] for i in order]
    mean_scores = [mean_scores[i] for i in order]
    bar_colors = ['#e05c5c' if r < 0.5 else '#f5a623' if r < 0.8 else '#4caf50' for r in recalls]
    bars = ax_pc.bar(classes, recalls, color=bar_colors, edgecolor='white', width=0.6)
    ax_pc.axhline(overall_defect_recall, color='navy', linestyle='--', lw=1.5, label=f'Overall recall = {overall_defect_recall:.1%}')
    ax_pc.axhline(0.8, color='gray', linestyle=':', lw=1, label='80% target')
    ax_pc.set_ylim(0, 1.12)
    ax_pc.set_ylabel('Detection Recall')
    ax_pc.set_title('Per-Class Defect Detection Recall  (red < 50% | orange 50-80% | green >= 80%)')
    ax_pc.legend(fontsize=9)
    for bar, rec, n, ms in zip(bars, recalls, counts, mean_scores):
        ax_pc.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f'{rec:.0%}\n(n={n})', ha='center', va='bottom', fontsize=9, fontweight='bold')
    plt.suptitle(f'PatchCore + MAE Fine-Tuned ViT-B/16  |  ROC-AUC={roc_auc:.4f}  |  AUPRC={auprc:.4f}  |  Threshold=q{THRESHOLD_QUANTILE:.2f}', fontsize=13, fontweight='bold')
    plt.savefig(EVAL_PLOT_PATH, dpi=130, bbox_inches='tight')
    plt.show()
    defect_recall_df = pd.DataFrame([{'failure_label': c, **v} for c, v in perclass_results.items()]).sort_values('recall').reset_index(drop=True)
    defect_recall_df.to_csv(os.path.join(RESULTS_DIR, 'defect_recall.csv'), index=False)
    display(defect_recall_df)
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "fig = plt.figure(figsize=(21, 10))\ngs = fig.add_gridspec(2, 4, hspace=0.4, wspace=0.35)\nax_dist = fig.add_subplot(gs[0, 0])\nax_roc = fig.add_subplot(gs[0, 1])\nax_pr = fig.add_subplot(gs[0, 2])\nax_cm = fig.add_subplot(gs[0, 3])\nax_pc = fig.add_subplot(gs[1, :])\nax_dist.hist(test_normal_scores, bins=50, alpha=0.7, color='steelblue', label='normal')\nax_dist.hist(test_defect_scores, bins=50, alpha=0.7, color='tomato', label='defect')\nax_dist.axvline(best_thresh, color='black', linestyle='--', label=f'threshold={best_thresh:.4f}')\nax_dist.set_title('anomaly score distributions')\nax_dist.set_xlabel('anomaly score')\nax_dist.set_ylabel('count')\nax_dist.legend(fontsize=9)\nax_roc.plot(fpr_arr, tpr_arr, color='darkorange', lw=2, label=f'roc-auc = {roc_auc:.4f}')\nax_roc.plot([0, 1], [0, 1], 'k--', alpha=0.4)\nop_fpr = (test_normal_scores > best_thresh).mean()\nop_tpr = (test_defect_scores > best_thresh).mean()\nax_roc.scatter([op_fpr], [op_tpr], color='red', zorder=5, label=f'operating point\\nfpr={op_fpr:.3f} tpr={op_tpr:.3f}')\nax_roc.set_title('roc curve')\nax_roc.set_xlabel('false positive rate')\nax_roc.set_ylabel('true positive rate')\nax_roc.legend(fontsize=9)\nbaseline = all_labels.mean()\nax_pr.plot(recall_arr, precision_arr, color='purple', lw=2, label=f'auprc = {auprc:.4f}')\nax_pr.axhline(baseline, color='gray', linestyle='--', lw=1, label=f'no-skill baseline = {baseline:.3f}')\nax_pr.set_title('precision-recall curve')\nax_pr.set_xlabel('recall')\nax_pr.set_ylabel('precision')\nax_pr.set_xlim(0, 1)\nax_pr.set_ylim(0, 1.02)\nax_pr.legend(fontsize=9)\ncm = confusion_matrix(all_labels, predictions)\nsns.heatmap(cm, annot=true, fmt='d', cmap='blues', xticklabels=['pred normal', 'pred defect'], yticklabels=['true normal', 'true defect'], ax=ax_cm)\nax_cm.set_title('confusion matrix')\nclasses = list(perclass_results.keys())\nrecalls = [perclass_results[c]['recall'] for c in classes]\ncounts = [perclass_results[c]['n'] for c in classes]\nmean_scores = [perclass_results[c]['mean_score'] for c in classes]\norder = np.argsort(recalls)\nclasses = [classes[i] for i in order]\nrecalls = [recalls[i] for i in order]\ncounts = [counts[i] for i in order]\nmean_scores = [mean_scores[i] for i in order]\nbar_colors = ['#e05c5c' if r < 0.5 else '#f5a623' if r < 0.8 else '#4caf50' for r in recalls]\nbars = ax_pc.bar(classes, recalls, color=bar_colors, edgecolor='white', width=0.6)\nax_pc.axhline(overall_defect_recall, color='navy', linestyle='--', lw=1.5, label=f'overall recall = {overall_defect_recall:.1%}')\nax_pc.axhline(0.8, color='gray', linestyle=':', lw=1, label='80% target')\nax_pc.set_ylim(0, 1.12)\nax_pc.set_ylabel('detection recall')\nax_pc.set_title('per-class defect detection recall  (red < 50% | orange 50-80% | green >= 80%)')\nax_pc.legend(fontsize=9)\nfor bar, rec, n, ms in zip(bars, recalls, counts, mean_scores):\n    ax_pc.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f'{rec:.0%}\\n(n={n})', ha='center', va='bottom', fontsize=9, fontweight='bold')\nplt.suptitle(f'patchcore + mae fine-tuned vit-b/16  |  roc-auc={roc_auc:.4f}  |  auprc={auprc:.4f}  |  threshold=q{threshold_quantile:.2f}', fontsize=13, fontweight='bold')\nplt.savefig(eval_plot_path, dpi=130, bbox_inches='tight')\nplt.show()\ndefect_recall_df = pd.dataframe([{'failure_label': c, **v} for c, v in perclass_results.items()]).sort_values('recall').reset_index(drop=true)\ndefect_recall_df.to_csv(os.path.join(results_dir, 'defect_recall.csv'), index=false)\ndisplay(defect_recall_df)\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## UMAP Visualization

This cell projects image-level embeddings (mean-pooled patch features) into 2D using UMAP. When `FORCE_RERUN_UMAP=False` and the saved figure exists, it displays the saved PNG directly.


In [ ]:
try:
    import subprocess
    from sklearn.decomposition import PCA
    UMAP_PNG_PATH = str(globals().get('UMAP_PNG_PATH', Path(PLOTS_DIR) / 'umap_test_embeddings.png'))
    UMAP_CSV_PATH = str(globals().get('UMAP_CSV_PATH', Path(RESULTS_DIR) / 'umap_test_embeddings.csv'))
    seed_value = int(globals().get('SEED', 42))
    if 'test_normal_scores_z' in globals():
        normal_scores = np.asarray(test_normal_scores_z)
        defect_scores = np.asarray(test_defect_scores_z)
        threshold_value = float(globals().get('threshold_z', globals().get('best_thresh')))
        score_column = 'score_z'
    else:
        normal_scores = np.asarray(test_normal_scores)
        defect_scores = np.asarray(test_defect_scores)
        threshold_value = float(globals().get('best_thresh', globals().get('threshold_z')))
        score_column = 'score'
    if 'test_normal_loader' not in globals() or 'test_defect_loader' not in globals():
        batch_size = int(globals().get('BATCH_SIZE', globals().get('BANK_BATCH_SIZE', 128)))
        num_workers = int(globals().get('NUM_WORKERS', globals().get('BANK_WORKERS', 0)))
        _loader_kw = dict(batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=USE_CUDA, persistent_workers=num_workers > 0)
        test_normal_loader = DataLoader(WaferDataset(test_normal_df, IMAGE_SIZE), **_loader_kw)
        test_defect_loader = DataLoader(WaferDataset(test_defect_df, IMAGE_SIZE), **_loader_kw)
    try:
        import umap.umap_ as umap
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'umap-learn', '-q'])
        import umap.umap_ as umap

    def collect_image_embeddings(loader, frame, split_name, desc):
        embeddings = []
        metadata_rows = []
        seen = 0
        with torch.inference_mode():
            for xb, yb in tqdm(loader, total=len(loader), desc=desc, unit='batch'):
                batch_size = len(yb)
                xb = xb.to(DEVICE)
                with torch.autocast('cuda', torch.float16, enabled=USE_CUDA):
                    feat = extractor(xb)
                    emb = extractor.proj(feat)
                img_emb = F.normalize(emb.float().mean(dim=1), p=2, dim=1)
                embeddings.append(img_emb.cpu().numpy())
                batch_meta = frame.iloc[seen:seen + batch_size][['failure_label', 'is_anomaly']].copy()
                batch_meta = batch_meta.reset_index(drop=True)
                batch_meta['split'] = split_name
                metadata_rows.append(batch_meta)
                seen += batch_size
        return (np.concatenate(embeddings, axis=0), pd.concat(metadata_rows, ignore_index=True))
    X_normal, meta_normal = collect_image_embeddings(test_normal_loader, test_normal_df, 'test_normal', 'Embed test-normal')
    X_defect, meta_defect = collect_image_embeddings(test_defect_loader, test_defect_df, 'test_defect', 'Embed test-defect')
    X = np.concatenate([X_normal, X_defect], axis=0)
    umap_df = pd.concat([meta_normal, meta_defect], ignore_index=True)
    umap_df.insert(0, 'point_index', np.arange(len(umap_df)))
    umap_df[score_column] = np.concatenate([normal_scores, defect_scores])
    umap_df['predicted_anomaly'] = (umap_df[score_column] > threshold_value).astype(int)
    umap_df['label_name'] = umap_df['is_anomaly'].map({0: 'normal', 1: 'defect'})
    n_pca = min(32, X.shape[1], X.shape[0] - 1)
    Xp = PCA(n_components=n_pca, random_state=seed_value).fit_transform(X)
    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric='cosine', random_state=seed_value, transform_seed=seed_value, low_memory=True)
    coords = reducer.fit_transform(Xp)
    umap_df['umap_1'] = coords[:, 0]
    umap_df['umap_2'] = coords[:, 1]
    fig, ax = plt.subplots(figsize=(8, 6))
    normal_mask = umap_df['is_anomaly'].to_numpy() == 0
    defect_mask = ~normal_mask
    ax.scatter(coords[normal_mask, 0], coords[normal_mask, 1], s=8, alpha=0.45, label='Normal', c='steelblue', linewidths=0)
    ax.scatter(coords[defect_mask, 0], coords[defect_mask, 1], s=8, alpha=0.6, label='Defect', c='tomato', linewidths=0)
    ax.set_title('UMAP of Test Image Embeddings')
    ax.set_xlabel('UMAP-1')
    ax.set_ylabel('UMAP-2')
    ax.legend()
    fig.tight_layout()
    fig.savefig(UMAP_PNG_PATH, dpi=160, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    umap_df.to_csv(UMAP_CSV_PATH, index=False)
    print(f'Saved UMAP figure: {UMAP_PNG_PATH}')
    print(f'Saved UMAP points: {UMAP_CSV_PATH}')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "import subprocess\nfrom sklearn.decomposition import pca\numap_png_path = str(globals().get('umap_png_path', path(plots_dir) / 'umap_test_embeddings.png'))\numap_csv_path = str(globals().get('umap_csv_path', path(results_dir) / 'umap_test_embeddings.csv'))\nseed_value = int(globals().get('seed', 42))\nif 'test_normal_scores_z' in globals():\n    normal_scores = np.asarray(test_normal_scores_z)\n    defect_scores = np.asarray(test_defect_scores_z)\n    threshold_value = float(globals().get('threshold_z', globals().get('best_thresh')))\n    score_column = 'score_z'\nelse:\n    normal_scores = np.asarray(test_normal_scores)\n    defect_scores = np.asarray(test_defect_scores)\n    threshold_value = float(globals().get('best_thresh', globals().get('threshold_z')))\n    score_column = 'score'\nif 'test_normal_loader' not in globals() or 'test_defect_loader' not in globals():\n    batch_size = int(globals().get('batch_size', globals().get('bank_batch_size', 128)))\n    num_workers = int(globals().get('num_workers', globals().get('bank_workers', 0)))\n    _loader_kw = dict(batch_size=batch_size, shuffle=false, num_workers=num_workers, pin_memory=use_cuda, persistent_workers=num_workers > 0)\n    test_normal_loader = dataloader(waferdataset(test_normal_df, image_size), **_loader_kw)\n    test_defect_loader = dataloader(waferdataset(test_defect_df, image_size), **_loader_kw)\ntry:\n    import umap.umap_ as umap\nexcept exception:\n    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'umap-learn', '-q'])\n    import umap.umap_ as umap\n\ndef collect_image_embeddings(loader, frame, split_name, desc):\n    embeddings = []\n    metadata_rows = []\n    seen = 0\n    with torch.inference_mode():\n        for xb, yb in tqdm(loader, total=len(loader), desc=desc, unit='batch'):\n            batch_size = len(yb)\n            xb = xb.to(device)\n            with torch.autocast('cuda', torch.float16, enabled=use_cuda):\n                feat = extractor(xb)\n                emb = extractor.proj(feat)\n            img_emb = f.normalize(emb.float().mean(dim=1), p=2, dim=1)\n            embeddings.append(img_emb.cpu().numpy())\n            batch_meta = frame.iloc[seen:seen + batch_size][['failure_label', 'is_anomaly']].copy()\n            batch_meta = batch_meta.reset_index(drop=true)\n            batch_meta['split'] = split_name\n            metadata_rows.append(batch_meta)\n            seen += batch_size\n    return (np.concatenate(embeddings, axis=0), pd.concat(metadata_rows, ignore_index=true))\nx_normal, meta_normal = collect_image_embeddings(test_normal_loader, test_normal_df, 'test_normal', 'embed test-normal')\nx_defect, meta_defect = collect_image_embeddings(test_defect_loader, test_defect_df, 'test_defect', 'embed test-defect')\nx = np.concatenate([x_normal, x_defect], axis=0)\numap_df = pd.concat([meta_normal, meta_defect], ignore_index=true)\numap_df.insert(0, 'point_index', np.arange(len(umap_df)))\numap_df[score_column] = np.concatenate([normal_scores, defect_scores])\numap_df['predicted_anomaly'] = (umap_df[score_column] > threshold_value).astype(int)\numap_df['label_name'] = umap_df['is_anomaly'].map({0: 'normal', 1: 'defect'})\nn_pca = min(32, x.shape[1], x.shape[0] - 1)\nxp = pca(n_components=n_pca, random_state=seed_value).fit_transform(x)\nreducer = umap.umap(n_components=2, n_neighbors=15, min_dist=0.1, metric='cosine', random_state=seed_value, transform_seed=seed_value, low_memory=true)\ncoords = reducer.fit_transform(xp)\numap_df['umap_1'] = coords[:, 0]\numap_df['umap_2'] = coords[:, 1]\nfig, ax = plt.subplots(figsize=(8, 6))\nnormal_mask = umap_df['is_anomaly'].to_numpy() == 0\ndefect_mask = ~normal_mask\nax.scatter(coords[normal_mask, 0], coords[normal_mask, 1], s=8, alpha=0.45, label='normal', c='steelblue', linewidths=0)\nax.scatter(coords[defect_mask, 0], coords[defect_mask, 1], s=8, alpha=0.6, label='defect', c='tomato', linewidths=0)\nax.set_title('umap of test image embeddings')\nax.set_xlabel('umap-1')\nax.set_ylabel('umap-2')\nax.legend()\nfig.tight_layout()\nfig.savefig(um"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Save Outputs

This cell exports the full PatchCore model artifact, evaluation metrics, and summary JSON for reproducibility.


In [ ]:
try:
    import shutil
    checkpoints_dir = Path(globals().get('CHECKPOINTS_DIR', globals().get('CHKPT_DIR', Path(ARTIFACT_DIR) / 'checkpoints')))
    plots_dir = Path(globals().get('PLOTS_DIR', Path(ARTIFACT_DIR) / 'plots'))
    results_dir = Path(globals().get('RESULTS_DIR', Path(ARTIFACT_DIR) / 'results'))
    checkpoints_dir.mkdir(parents=True, exist_ok=True)
    plots_dir.mkdir(parents=True, exist_ok=True)
    results_dir.mkdir(parents=True, exist_ok=True)
    MODEL_EXPORT_PATH = str(globals().get('MODEL_EXPORT_PATH', checkpoints_dir / 'model.pt'))
    METRICS_EXPORT_PATH = str(globals().get('METRICS_EXPORT_PATH', results_dir / 'evaluation_metrics.json'))
    TEST_SCORES_CSV_PATH = str(globals().get('TEST_SCORES_CSV_PATH', results_dir / 'test_scores.csv'))
    DEFECT_RECALL_CSV_PATH = str(globals().get('DEFECT_RECALL_CSV_PATH', results_dir / 'test_defect_recall.csv'))
    UMAP_PNG_PATH = str(globals().get('UMAP_PNG_PATH', plots_dir / 'umap_test_embeddings.png'))
    UMAP_CSV_PATH = str(globals().get('UMAP_CSV_PATH', results_dir / 'umap_test_embeddings.csv'))
    threshold_value = float(globals().get('threshold_z', globals().get('best_thresh', 0.0)))
    threshold_raw_value = float(globals().get('threshold_raw', threshold_value))
    artifact = {'threshold_z': threshold_value, 'threshold_raw': threshold_raw_value, 'config': {}}
    for key in ['IMAGE_SIZE', 'VIT_FEATURE_BLOCK', 'VIT_FEATURE_BLOCKS', 'PATCH_EMBED_DIM', 'TOPK_PATCH_RATIO', 'PATCHCORE_NN_K']:
        if key in globals():
            artifact['config'][key.lower()] = globals()[key]
    if 'mu' in globals():
        artifact['train_score_mu'] = float(mu)
    if 'std' in globals():
        artifact['train_score_std'] = float(std)
    if 'extractor' in globals():
        artifact['extractor_state_dict'] = extractor.state_dict()
    elif 'model' in globals() and hasattr(model, 'state_dict'):
        artifact['model_state_dict'] = model.state_dict()
    if 'memory_bank' in globals() and memory_bank is not None:
        artifact['memory_bank'] = memory_bank.cpu()
    elif Path(MODEL_EXPORT_PATH).exists():
        try:
            existing_artifact = torch.load(MODEL_EXPORT_PATH, map_location='cpu')
            if isinstance(existing_artifact, dict):
                for key in ['memory_bank', 'extractor_state_dict', 'model_state_dict']:
                    if key in existing_artifact and key not in artifact:
                        artifact[key] = existing_artifact[key]
        except Exception:
            pass
    if artifact:
        torch.save(artifact, MODEL_EXPORT_PATH)
    if 'test_scores_df' in globals():
        test_scores_df.to_csv(TEST_SCORES_CSV_PATH, index=False)
    if 'defect_recall_df' in globals():
        defect_recall_df.to_csv(DEFECT_RECALL_CSV_PATH, index=False)
    elif 'defect_df_out' in globals():
        defect_df_out.rename(columns={'failure_label': 'failure_label'}).to_csv(DEFECT_RECALL_CSV_PATH, index=False)
    elif 'breakdown' in globals():
        breakdown.to_csv(DEFECT_RECALL_CSV_PATH, index=False)
    metrics_payload = {'threshold_z': threshold_value, 'threshold_raw': threshold_raw_value, 'model_export_path': MODEL_EXPORT_PATH, 'test_scores_csv_path': TEST_SCORES_CSV_PATH, 'defect_recall_csv_path': DEFECT_RECALL_CSV_PATH, 'umap_png_path': UMAP_PNG_PATH, 'umap_csv_path': UMAP_CSV_PATH}
    if 'roc_auc' in globals():
        metrics_payload['roc_auc_z'] = float(roc_auc)
    if 'auroc' in globals():
        metrics_payload['roc_auc_z'] = float(auroc)
    if 'cm' in globals():
        metrics_payload['confusion_matrix'] = cm.tolist()
    Path(METRICS_EXPORT_PATH).write_text(json.dumps(metrics_payload, indent=2), encoding='utf-8')
    print(f'Saved model artifact: {MODEL_EXPORT_PATH}')
    print(f'Saved metrics: {METRICS_EXPORT_PATH}')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "import shutil\ncheckpoints_dir = path(globals().get('checkpoints_dir', globals().get('chkpt_dir', path(artifact_dir) / 'checkpoints')))\nplots_dir = path(globals().get('plots_dir', path(artifact_dir) / 'plots'))\nresults_dir = path(globals().get('results_dir', path(artifact_dir) / 'results'))\ncheckpoints_dir.mkdir(parents=true, exist_ok=true)\nplots_dir.mkdir(parents=true, exist_ok=true)\nresults_dir.mkdir(parents=true, exist_ok=true)\nmodel_export_path = str(globals().get('model_export_path', checkpoints_dir / 'model.pt'))\nmetrics_export_path = str(globals().get('metrics_export_path', results_dir / 'evaluation_metrics.json'))\ntest_scores_csv_path = str(globals().get('test_scores_csv_path', results_dir / 'test_scores.csv'))\ndefect_recall_csv_path = str(globals().get('defect_recall_csv_path', results_dir / 'test_defect_recall.csv'))\numap_png_path = str(globals().get('umap_png_path', plots_dir / 'umap_test_embeddings.png'))\numap_csv_path = str(globals().get('umap_csv_path', results_dir / 'umap_test_embeddings.csv'))\nthreshold_value = float(globals().get('threshold_z', globals().get('best_thresh', 0.0)))\nthreshold_raw_value = float(globals().get('threshold_raw', threshold_value))\nartifact = {'threshold_z': threshold_value, 'threshold_raw': threshold_raw_value, 'config': {}}\nfor key in ['image_size', 'vit_feature_block', 'vit_feature_blocks', 'patch_embed_dim', 'topk_patch_ratio', 'patchcore_nn_k']:\n    if key in globals():\n        artifact['config'][key.lower()] = globals()[key]\nif 'mu' in globals():\n    artifact['train_score_mu'] = float(mu)\nif 'std' in globals():\n    artifact['train_score_std'] = float(std)\nif 'extractor' in globals():\n    artifact['extractor_state_dict'] = extractor.state_dict()\nelif 'model' in globals() and hasattr(model, 'state_dict'):\n    artifact['model_state_dict'] = model.state_dict()\nif 'memory_bank' in globals() and memory_bank is not none:\n    artifact['memory_bank'] = memory_bank.cpu()\nelif path(model_export_path).exists():\n    try:\n        existing_artifact = torch.load(model_export_path, map_location='cpu')\n        if isinstance(existing_artifact, dict):\n            for key in ['memory_bank', 'extractor_state_dict', 'model_state_dict']:\n                if key in existing_artifact and key not in artifact:\n                    artifact[key] = existing_artifact[key]\n    except exception:\n        pass\nif artifact:\n    torch.save(artifact, model_export_path)\nif 'test_scores_df' in globals():\n    test_scores_df.to_csv(test_scores_csv_path, index=false)\nif 'defect_recall_df' in globals():\n    defect_recall_df.to_csv(defect_recall_csv_path, index=false)\nelif 'defect_df_out' in globals():\n    defect_df_out.rename(columns={'failure_label': 'failure_label'}).to_csv(defect_recall_csv_path, index=false)\nelif 'breakdown' in globals():\n    breakdown.to_csv(defect_recall_csv_path, index=false)\nmetrics_payload = {'threshold_z': threshold_value, 'threshold_raw': threshold_raw_value, 'model_export_path': model_export_path, 'test_scores_csv_path': test_scores_csv_path, 'defect_recall_csv_path': defect_recall_csv_path, 'umap_png_path': umap_png_path, 'umap_csv_path': umap_csv_path}\nif 'roc_auc' in globals():\n    metrics_payload['roc_auc_z'] = float(roc_auc)\nif 'auroc' in globals():\n    metrics_payload['roc_auc_z'] = float(auroc)\nif 'cm' in globals():\n    metrics_payload['confusion_matrix'] = cm.tolist()\npath(metrics_export_path).write_text(json.dumps(metrics_payload, indent=2), encoding='utf-8')\nprint(f'saved model artifact: {model_export_path}')\nprint(f'saved metrics: {metrics_export_path}')\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Cleanup

This cell frees GPU memory and large in-memory objects after all artifacts have been saved.


In [ ]:
for name in ['mae_model', 'extractor', 'memory_bank', 'tune_normal_scores', 'test_normal_scores', 'test_defect_scores', 'all_scores', 'all_labels', 'predictions', 'train_loader', 'tune_normal_loader', 'test_normal_loader', 'test_defect_loader']:
    if name in globals():
        del globals()[name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
print('Cleared.')
